# 🎯 [AMM02_M02] 부대 규모, 교전 강도, 이동 거리 변수 정규화 및 소모량 산출 전처리 모듈 구현 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/AMM02/amm02_m02_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 탄약·특수물자 (Level 3) |
| **미션 ID** | AMM02_M02 |
| **미션 주제** | 부대 규모, 교전 강도, 이동 거리 변수 정규화 및 소모량 산출 전처리 모듈 구현 |
| **소요 시간** | 약 2시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
탄약 저장고는 공간이 한정되어 있고 품목마다 함께 둘 수 없는 제약이 있어, 앞으로 얼마나 소모되고 얼마나 자주 반출될지를 알아야 입고 위치와 재보급 순서를 합리적으로 정할 수 있다. 그러나 작전 단계별 고정 기준 소모율(CSR)만으로 계획하면 교전 강도가 낮은 날은 과잉 재고가 쌓이고 높은 날은 긴급 재보급이 반복되어 저장 공간과 작업 동선이 낭비된다. 본 프로젝트는 교육용 가상 부대·가상 품목·가상 저장고 자료를 코드로 만들어, 탄약·유류 소모율 데이터 정의와 전처리, CSR 기반 Baseline과 탐색적 분석, 적 저항·화력 계획을 반영한 소모량 예측, 보급 수준(DOS) 연계 재보급 산출, 작전 지도 대시보드와 설명가능성, 저장 배치 최적화와 군수참모 결재선(HITL), 7일 고강도 스트레스 시험, AAR 기반 소모 계수 재보정을 단계별로 구현한다. 최종 목표는 예측 소모량을 근거로 재보급 소요와 저장 위치를 함께 제안해 저장 효율과 작업 동선을 개선하고, 불출·대체 탄종·재배치는 군수참모가 근거를 확인해 결정하도록 지원하는 통합 시뮬레이터를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 야전(군수참모처, 탄약창)에 즉시 도입 가능한 "완성형 국방 AI 시뮬레이터 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 행정 및 기술 명세서** | 도메인 데이터 명세서(`docs/amm02_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/amm02_operations_guide.md`) | 미션 1, 10 |
| **`src/etl/`** | **군수 데이터 정제 모듈** | 10종 정제 룰 및 긴급 불출 보존 ETL 파이프라인(`src/etl/amm02_data_cleansing.py`) | 미션 2 |
| **`notebooks/`** | **분석 및 통계 검증** | 90일 소모량 Baseline 비교, 가설 검정 및 ABC 파레토 분석(`notebooks/amm02_eda_analysis.ipynb`) | 미션 3 |
| **`src/features/`** | **전술 피처 엔지니어링** | 시계열 파생 피처 엔지니어링 모듈(`src/features/amm02_feature_engineering.py`) | 미션 4 |
| **`src/models/`** | **AI 예측 & 재보급 엔진** | GBR+RF 앙상블 및 DOS 재보급 산출 엔진(`src/models/amm02_advanced_model.py`) | 미션 5 |
| **`src/explainability/`** | **XAI 설명가능성 모듈** | XAI 기여도 분석 및 자연어 보고서 생성기(`src/explainability/amm02_xai_explainer.py`) | 미션 6 |
| **`src/optimization/`** | **저장 최적화 & HITL** | 정수계획 저장 최적화 및 참모 HITL 결재선 엔진(`src/optimization/amm02_simulator.py`) | 미션 7 |
| **`src/dashboard/`** | **작전 지도 대시보드** | 전술 격자 지도 연동 인터랙티브 대시보드 웹앱(`src/dashboard/amm02_app.py`) | 미션 8 |
| **`reports/`** | **검증 및 AAR 리포트** | 사후검토(AAR) 편차 분석, 축소 추정 CSR 재보정 및 3대 정량 효과 리포트(`reports/amm02_roi_analysis.md`) | 미션 9 |
| **`pipeline/`** | **MLOps 자동 재학습** | 신규 훈련 데이터 기반 챔피언/챌린저 모델 자동 성능 비교 및 롤오버 파이프라인(`pipeline/amm02_retrain.py`) | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. 부대 규모, 교전 강도, 이동 거리 변수 정규화 및 소모량 산출 전처리 모듈 구현
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

부대 규모, 교전 강도, 이동 거리 변수 정규화 전처리 모듈을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **부대 규모, 교전 강도, 이동 거리 변수를 정제·정규화하고 병력 100명당 환산 소모율을 산출하는 전처리 모듈을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **중복 기록 제거, 표기 통일, 백분율·자릿수 오기 보정, 음수·이상치 결측 처리, KNN 결측 보정 등 정제 규칙 10종을 적용 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/etl/amm02_data_cleansing.py 및 정제 품질 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 특수 예외(비계획 정비, 긴급 불출 등) 데이터 보존의 타당성

#### 미션 PASS 조건

- 정량 기준: 데이터 정제 규칙 10종 적용, 데이터 정합성 검증 통과율 100%


- 진행 기준: ETL 파이프라인 구현 및 정제된 마스터 데이터셋 구축


- 검증 방법: 전후 데이터 통계량(기술통계) 비교 및 무결성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
# [TODO : 학생 실습 2] 부대 규모, 교전 강도, 이동 거리 변수 정규화 및 소모량 산출 전처리 모듈 구현
# 미션 지침 및 검증 기준을 확인하고 코드를 실행 및 튜닝하세요.
"""부대 규모·교전 강도·이동 거리 변수를 정제·정규화하고, 병력 100명당 환산 소모율을 산출하는 전처리 모듈을 구현한다. 정제 규칙 10종을 적용하고 긴급 불출 같은 군수 예외 기록은 삭제하지 않고 보존하는지 검증한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 부대·가상 품목 일일 소모 기록 생성기 (실제 탄약 제원·소모율과 무관)
# 가상 부대 8개 x 90일. 소모량 = 가상 CSR 계수 x 병력 규모 x (교전 강도·적 저항·화력 계획 효과) x 잡음
# ---------------------------------------------------------------------------
UNITS = [f"U{i:02d}" for i in range(1, 9)]
ITEMS = ["ammo_A", "ammo_B", "ammo_C", "ammo_D", "fuel_jet", "fuel_diesel"]
AMMO, FUEL = ITEMS[:4], ITEMS[4:]
PHASES = ["prep", "attack", "defense", "stability"]
CSR = {  # 가상 계수: 병력 100명·1일 기준 소모 단위(탄약=상자, 유류=드럼)
    "prep":      [1.0, 0.6, 0.3, 0.2, 1.5, 6.0],
    "attack":    [6.0, 3.5, 2.0, 1.2, 5.0, 12.0],
    "defense":   [4.5, 3.0, 1.6, 0.8, 3.0, 7.0],
    "stability": [0.8, 0.5, 0.3, 0.2, 1.0, 5.0],
}
PHASE_RESIST = {"prep": 1.0, "attack": 3.2, "defense": 3.6, "stability": 0.8}


def make_ops(n_days=90, seed=7, surge=0):
    """가상 부대별 일일 작전 기록과 품목별 실제 소모량(정답)을 만든다. surge>0이면 마지막 surge일을 고강도 공격 단계로 바꾼다(스트레스 시험용)."""
    rng = np.random.default_rng(seed)
    rows = []
    for uid in UNITS:
        size = int(rng.integers(3, 9)) * 100
        lens = rng.integers(8, 20, 4)
        sched = ["prep"] * lens[0] + ["attack"] * lens[1] + ["defense"] * lens[2] + ["attack"] * lens[3]
        sched = (sched + ["stability"] * n_days)[:n_days]
        if surge:
            sched[-surge:] = ["attack"] * surge
        r = 1.0
        for d, ph in enumerate(sched):
            r = float(np.clip(0.7 * r + 0.3 * (4.8 if surge and d >= n_days - surge else PHASE_RESIST[ph]) + rng.normal(0, 0.35), 0, 5))
            inten = float(np.clip(r / 5 * 0.9 + rng.normal(0, 0.06), 0, 1))
            fire = int(rng.poisson(1 + 8 * inten))
            move = float(rng.gamma(2, {"prep": 4, "attack": 12, "defense": 2, "stability": 5}[ph]))
            weather = str(rng.choice(["clear", "rain", "snow"], p=[0.7, 0.2, 0.1]))
            emerg = bool(rng.random() < 0.02)
            rec = {"unit": uid, "day": d, "phase": ph, "unit_size": size, "intensity": round(inten, 3),
                   "resistance": round(r, 2), "fire_missions": fire, "move_km": round(move, 1),
                   "weather": weather, "emergency": emerg}
            base = size / 100
            for i, it in enumerate(ITEMS):
                c = CSR[ph][i] * base
                if it in AMMO:
                    c *= (0.5 + 0.6 * inten) * (1 + 0.04 * fire * (1.0 if it in ("ammo_C", "ammo_D") else 0.4))
                elif it == "fuel_jet":
                    c *= (0.6 + 0.5 * inten) + 0.03 * fire
                else:
                    c = c * (0.8 + 0.2 * (weather != "clear")) + 0.05 * move * base
                c *= rng.lognormal(0, 0.12) * (2.5 if emerg else 1.0)
                rec[it] = round(float(c), 1)
            rows.append(rec)
    return pd.DataFrame(rows)
# ---------------------------------------------------------------------------


# 1) [교육용가상자료] 원천 기록에 현장 입력 오류를 일부러 섞는다
clean_truth = make_ops(90, seed=7)
raw = clean_truth.copy()
rng = np.random.default_rng(3)
n = len(raw)
idx = lambda k: rng.choice(n, k, replace=False)
raw = raw.astype({"phase": object, "weather": object, "unit_size": float, "fire_missions": float})
raw.loc[idx(15), "phase"] = raw["phase"].str.upper()                          # 표기 불일치
raw.loc[idx(20), "weather"] = "비"                                            # 한글 표기 혼용
raw.loc[idx(12), "intensity"] = raw["intensity"] * 100                         # 백분율로 입력
raw.loc[idx(10), "ammo_A"] = -raw["ammo_A"]                                   # 부호 오류
raw.loc[idx(6), "fire_missions"] = -1
raw.loc[idx(8), "move_km"] = 999.0                                            # 계기 오류
raw.loc[idx(18), "intensity"] = np.nan                                        # 결측
raw.loc[idx(14), "move_km"] = np.nan
raw.loc[idx(5), "unit_size"] = raw["unit_size"] * 10                          # 단위 오기(자릿수)
raw = pd.concat([raw, raw.sample(12, random_state=1)]).sample(frac=1, random_state=2).reset_index(drop=True)  # 중복·순서 섞임
n_emerg = int(clean_truth["emergency"].sum())

# 2) 정제 규칙 10종
log = []
df = raw.copy()
before = len(df); df = df.drop_duplicates(["unit", "day"]); log.append(("R01", "부대·일자 중복 기록 제거", before - len(df)))
m = df["phase"] != df["phase"].str.strip().str.lower(); df["phase"] = df["phase"].str.strip().str.lower(); log.append(("R02", "작전 단계 표기 통일", int(m.sum())))
wmap = {"비": "rain", "눈": "snow", "맑음": "clear"}; m = df["weather"].isin(wmap); df["weather"] = df["weather"].replace(wmap); log.append(("R03", "기상 표기 통일(한글->코드)", int(m.sum())))
m = df["intensity"] > 1; df.loc[m, "intensity"] /= 100; log.append(("R04", "교전 강도 백분율 입력 환산", int(m.sum())))
m = df["unit_size"] > 3000; df.loc[m, "unit_size"] /= 10; log.append(("R05", "병력 규모 자릿수 오기 보정", int(m.sum())))
m = df[ITEMS].lt(0).any(axis=1) | df["fire_missions"].lt(0)
df[ITEMS] = df[ITEMS].mask(df[ITEMS] < 0); df["fire_missions"] = df["fire_missions"].mask(df["fire_missions"] < 0); log.append(("R06", "음수 소모량·화력 임무 결측 처리", int(m.sum())))
m = (df["move_km"] > 150) & ~df["emergency"]; df.loc[m, "move_km"] = np.nan; log.append(("R07", "이동 거리 이상치(150km 초과) 결측 처리", int(m.sum())))
num = ["unit_size", "intensity", "resistance", "fire_missions", "move_km"] + ITEMS
n_missing = int(df[num].isna().any(axis=1).sum())
mu, sd = df[num].mean(), df[num].std()
df[num] = KNNImputer(n_neighbors=5).fit_transform((df[num] - mu) / sd) * sd.values + mu.values
df["fire_missions"] = df["fire_missions"].round(); log.append(("R08", "결측값 KNN 보정(유사 기록 5건)", n_missing))
df = df.sort_values(["unit", "day"]).reset_index(drop=True)
gaps = int(df.groupby("unit")["day"].apply(lambda s: (s.diff().dropna() != 1).sum()).sum()); log.append(("R09", "부대별 일자 정렬·연속성 확인(누락일)", gaps))
kept = int(df["emergency"].sum()); log.append(("R10", "긴급 불출 예외 기록 보존(삭제 금지)", kept))
print(pd.DataFrame(log, columns=["규칙", "내용", "처리 건수"]).to_string(index=False))

# 3) 정규화와 환산 소모율 산출
for c in ["unit_size", "intensity", "move_km"]:
    df[c + "_n"] = (df[c] - df[c].min()) / (df[c].max() - df[c].min())
for it in ITEMS:
    df[it + "_rate"] = df[it] / (df["unit_size"] / 100)          # 병력 100명·1일 환산 소모율

# 4) 정합성 검증
checks = {
    "중복 0건": not df.duplicated(["unit", "day"]).any(),
    "결측 0건": not df[num].isna().any().any(),
    "작전 단계 허용값": df["phase"].isin(PHASES).all(),
    "기상 허용값": df["weather"].isin(["clear", "rain", "snow"]).all(),
    "교전 강도 0~1": df["intensity"].between(0, 1).all(),
    "소모량 0 이상": (df[ITEMS] >= 0).all().all(),
    "병력 규모 복원 일치": (df["unit_size"].values == clean_truth.sort_values(["unit", "day"])["unit_size"].values).all(),
    "정규화 범위 0~1": df[[c + "_n" for c in ["unit_size", "intensity", "move_km"]]].stack().between(0, 1).all(),
    "일자 연속성": gaps == 0,
    "긴급 불출 기록 보존": kept == n_emerg,
}
rate = np.mean(list(checks.values()))
print(f"\n정합성 검증 {sum(checks.values())}/{len(checks)} 통과, 통과율 {rate:.0%} (목표 100%: {'달성' if rate == 1 else '미달'})")
print(f"정제 규칙 {len(log)}종 적용, 원천 {len(raw)}건 -> 정제 {len(df)}건, 긴급 불출 {kept}건 보존")

# 5) 전후 기술통계 비교
cols = ["unit_size", "intensity", "move_km", "ammo_A"]
comp = pd.DataFrame({"정제 전 평균": raw[cols].mean(), "정제 후 평균": df[cols].mean(),
                     "정제 전 최대": raw[cols].max(), "정제 후 최대": df[cols].max()}).round(2)
print("\n[정제 전후 기술통계]")
print(comp.to_string())
print("\n[작전 단계별 환산 소모율 평균(100명·1일, 가상 단위)]")
print(df.groupby("phase")[[it + "_rate" for it in ITEMS]].mean().round(2).rename(columns=lambda c: c[:-5]).to_string())

fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
for a, c in zip(ax, ["intensity", "move_km", "unit_size"]):
    a.hist(raw[c].dropna(), bins=30, alpha=0.5, label="raw"); a.hist(df[c], bins=30, alpha=0.7, label="cleaned")
    a.set_title(c); a.set_yscale("log"); a.legend()
plt.suptitle("AMM02 preprocessing: raw vs cleaned (synthetic)"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
 규칙                        내용  처리 건수
R01            부대·일자 중복 기록 제거     12
R02               작전 단계 표기 통일     15
R03          기상 표기 통일(한글->코드)     20
R04           교전 강도 백분율 입력 환산     12
R05           병력 규모 자릿수 오기 보정      5
R06        음수 소모량·화력 임무 결측 처리     16
R07 이동 거리 이상치(150km 초과) 결측 처리      8
R08      결측값 KNN 보정(유사 기록 5건)     54
R09     부대별 일자 정렬·연속성 확인(누락일)      0
R10     긴급 불출 예외 기록 보존(삭제 금지)     13

정합성 검증 10/10 통과, 통과율 100% (목표 100%: 달성)
정제 규칙 10종 적용, 원천 732건 -> 정제 720건, 긴급 불출 13건 보존

[정제 전후 기술통계]
           정제 전 평균  정제 후 평균  정제 전 최대  정제 후 최대
unit_size   688.66   650.00   8000.0   800.00
intensity     0.89     0.35     74.4     0.89
move_km      23.52    12.51    999.0    93.10
ammo_A       16.89    17.27    142.1   142.10

[작전 단계별 환산 소모율 평균(100명·1일, 가상 단위)]
           ammo_A  ammo_B  ammo_C  ammo_D  fuel_jet  fuel_diesel
phase                                                           
attack       5.63    3.24    2.07    1.25      5.36        11.93
defense      4.61    3.08    1.82    0.93      3.55         6.58
prep         0.64    0.39    0.20    0.14      1.18         5.66
stability    0.50    0.32    0.20    0.13      0.77         4.86
```
</details>
